Will be using the network in EI_STDP.ipynb where STDP is implemented on the synapses to the inhibitory neuron.
This seems to help the network find a steady state.

Here, we will explicitly compare the autaptic and non-autaptic networks in achieving synchrony.

In [ ]:
''' import aqua '''
from aqua.batchAQUA_general import batchAQUA
from aqua.AQUA_general import AQUA
from aqua.utils import * 

'''general imports''' 
import numpy as np
import pandas as pd
from brian2 import *
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
I_neuron = {'name': 'FS', 'C': 20, 'k': 1, 'v_r': -55, 'v_t': -40, 'v_peak': 25,
     'a': 0.2, 'b': -2, 'c': -45, 'd': 0, 'e': 0.2, 'f': 0., 'tau': 0.}

# strong autaptic neuron on RS resonator...
E1_neuron = {'name': 'RS', 'C': 100, 'k': 0.7, 'v_r': -60, 'v_t': -40, 'v_peak': 35,
     'a': 0.03, 'b': 5, 'c': -50, 'd': 100, 'e': 0.2, 'f': 250., 'tau': 0.}     # instantaneous autapse bc all synapses are instant.

# non-autaptic neuron - RS resonator
E2_neuron = {'name': 'RS', 'C': 100, 'k': 0.7, 'v_r': -60, 'v_t': -40, 'v_peak': 35,
     'a': 0.03, 'b': 5, 'c': -50, 'd': 100, 'e': 0., 'f': 0., 'tau': 0.}

def visualise_connectivity(S):
    Ns = len(S.source)
    Nt = len(S.target)
    figure(figsize=(10, 4))
    subplot(121)
    plot(zeros(Ns), arange(Ns), 'ok', ms=10)
    plot(ones(Nt), arange(Nt), 'ok', ms=10)
    for i, j in zip(S.i, S.j):
        plot([0, 1], [i, j], '-k')
    xticks([0, 1], ['Source', 'Target'])
    ylabel('Neuron index')
    xlim(-0.1, 1.1)
    ylim(-1, max(Ns, Nt))
    subplot(122)
    plot(S.i, S.j, 'ok')
    xlim(-1, Ns)
    ylim(-1, Nt)
    xlabel('Source neuron index')
    ylabel('Target neuron index')

In [ ]:
'''
- - - working 3 neuron circuit - - -

Try to show coherence and shunting inhibition.

This one features an autapse on one of the neurons.

'''
start_scope()

# simulation parameters
T = 5000 # ms
dt = 0.1
N_iter = int(T/dt)

# define the synapse at the population level, defined for excitatory and inhibitory populations
syn_eq = """
    dSyn_exc/dt = -(Syn_exc/t_exc)/ms : 1 
    t_exc : 1
    dSyn_inh/dt = -(Syn_inh/t_inh)/ms : 1 
    t_inh : 1
    g_total = Syn_exc + Syn_inh : 1 

"""

''' - - - define the excitatory population - - - '''
# neuron parameters
params_E = [E1_neuron, E2_neuron]      # 2 neurons

x_start = np.full(shape = (2, 3), fill_value = np.array([-60, 0, 0]))
t_start = np.zeros(2)

# create the batch
batch_E = batchAQUA(params_E)
batch_E.Initialise(x_start, t_start)

# create the input current - STEP CURRENT
I_h1 = 220                          # driving current to autaptic neuron
I1 = I_h1 * np.ones(N_iter)         # stronger driving current to E1s
I_h2 = 180                          # driving current to non-autaptic neuron
I2 = I_h2 * np.ones(N_iter)         # weaker current to E2

I_inj = np.array([I1, I2])
I_excTA = TimedArray(values = I_inj.T, dt = dt*ms, name = 'I_excTA')

# convert to brian2 with the standard autapse model
E, aut_E = batch_E.meetBrian(stimulus_name = I_excTA, synapse_eq = syn_eq)

''' - - - define the inhibitory neuron - - - '''
param_I = [I_neuron]
x_start = np.array([[-60, 0, 0]])
t_start = np.array([0.])

# create batch 
batch_I = batchAQUA(param_I)
batch_I.Initialise(x_start, t_start)

# input current will be just subthreshold
threshold, _ = batch_I.get_threshold(idx = 0)
I_inh = np.array([threshold*np.ones(N_iter)])
I_inhTA = TimedArray(values = I_inh.T, dt = dt*ms, name = 'I_inhTA')

# create brian objects, no effective autapse here.
I, aut_I = batch_I.meetBrian(stimulus_name = I_inhTA, synapse_eq = syn_eq)


''' - - - create the synapses - - - '''
# create the synapses between E1 and E2...
# I_syn represents the synapse strength
#syn_eq = """
#dI_syn/dt = -(I_syn/t_syn)/ms : 1 (clock-driven)
#t_syn : 1
#w_syn : 1
#g_total_syn = I_syn : 1 
#"""
model_exc = '''w_exc : 1'''
model_inh = '''w_inh : 1'''
syn_on_pre_exc = '''Syn_exc += w_exc'''     # excitatory presynaptic neuron
syn_on_pre_inh = '''Syn_inh += w_inh'''     # inhibitory presynaptic neuron

""" - - STDP - - """
w_max = 150 # maximum allowed current through an inhibitory synapse
taupre = taupost = 20 * ms
dApre = 20      # the maximum change in the weight in one step
dApost = -dApre * (taupre/taupost) * 1.05


model_stdp = '''
w_inh : 1
w_exc : 1
dApre/dt = -Apre / taupre : 1 (event-driven)
dApost/dt = -Apost / taupost : 1 (event-driven)
'''
# if pre- is inhibitory
on_pre_stdp_inh = '''
Syn_inh += w_inh
Apre += dApre
w_inh = clip(w_inh + Apost, -w_max, 0)
''' 
# if pre- is excitatory
on_pre_stdp_exc = '''
Syn_exc += w_exc
Apre += dApre
w_exc = clip(w_exc + Apost, 0, w_max)
''' 
on_post_stdp = '''
Apost += dApost
w_inh = clip(w_inh + Apre, 0, w_max)
w_exc = clip(w_exc + Apre, 0, w_max)
'''

'''exc. synapses'''
# fully connect excitatory neurons (ignoring autapses)
syn_E = Synapses(E, E, 
                model = model_exc,
                on_pre = syn_on_pre_exc,
                method = 'rk2')
syn_E.connect(condition = 'i != j')     # fully connected minus autapses

## Set exc. synapse variables here...
E.Syn_exc = 0            # pA
E.t_exc = 5              # ms
syn_E.w_exc[0, 1] = 60    # pA, weight from E1 -> E2
syn_E.w_exc[1, 0] = 60    # pA, weight from E2 -> E1

'''inh. to exc. synapses'''
syn_IE = Synapses(I, E,
                model = model_stdp,
                on_pre = on_pre_stdp_inh,
                on_post = on_post_stdp,
                method = 'rk2')
syn_IE.connect()         # fully connected to the 2 exc. neurons

## set inh. synapse variables for post-synaptic population
E.Syn_inh = 0
E.t_inh = 5                # ms
syn_IE.w_inh[0, 0] = -50   # pA, weight from I -> E1
syn_IE.w_inh[0, 1] = -50   # pA, weight from I -> E2
# alt., syn_I.w_syn[0, :] = 50, as they are equal

'''exc. to inh. synapses'''
syn_EI = Synapses(E, I,
                model = model_stdp,
                on_pre = on_pre_stdp_exc,
                on_post = on_post_stdp,
                method = 'rk2')
syn_EI.connect()         # both excitatory neurons connect to I

## set inh. synapse variables for post-syn population
I.Syn_exc = 0.            # initially no current through the synapse
I.t_exc = 5               # ms
I.t_inh = 5
syn_EI.w_exc[0, 0] = 50   # pA, weight from I -> E1
syn_EI.w_exc[1, 0] = 50   # pA, weight from I -> E2
# alt., syn_I.w_syn[0, :] = 50, as they are equal

''' - - simulation - - '''
# set simulation parameters
defaultclock.dt = dt*ms
# Monitors
M_v_E_aut = StateMonitor(E, ['v', 'Syn_exc', 'Syn_inh', 'w'], record = True)
M_v_I_aut = StateMonitor(I, ['v', 'Syn_exc', 'Syn_inh'], record = True)
spikemon_E_aut = SpikeMonitor(E, record = True)
spikemon_I_aut = SpikeMonitor(I, record = True)
# M_syn_E = StateMonitor(syn_E, 'Syn_exc', record = True)   # record the current through the synapse
# M_syn_IE = StateMonitor(syn_IE, 'Syn_inh', record = True)   
# M_syn_EI = StateMonitor(syn_EI, 'Syn_exc', record = True)   

# create networks
net = Network(E, I, aut_E, aut_I, syn_E, syn_IE, syn_EI, M_v_E, M_v_I, spikemon_E, spikemon_I) #, M_syn_E, M_syn_IE, M_syn_EI)

net.run(T*ms)

spike_trains_E = spikemon_E_aut.spike_trains()
spike_trains_I = spikemon_I_aut.spike_trains()

In [ ]:
'''
- - - working 3 neuron circuit - - -

Try to show coherence and shunting inhibition.

This one features no autapses.

'''
start_scope()

# simulation parameters
T = 5000 # ms
dt = 0.1
N_iter = int(T/dt)

# define the synapse at the population level, defined for excitatory and inhibitory populations
syn_eq = """
    dSyn_exc/dt = -(Syn_exc/t_exc)/ms : 1 
    t_exc : 1
    dSyn_inh/dt = -(Syn_inh/t_inh)/ms : 1 
    t_inh : 1
    g_total = Syn_exc + Syn_inh : 1 

"""

''' - - - define the excitatory population - - - '''
# neuron parameters
params_E = [E2_neuron, E2_neuron]      # 2 neurons

x_start = np.full(shape = (2, 3), fill_value = np.array([-60, 0, 0]))
t_start = np.zeros(2)

# create the batch
batch_E = batchAQUA(params_E)
batch_E.Initialise(x_start, t_start)

# create the input current - STEP CURRENT
I_h1 = 220                          # driving current to autaptic neuron
I1 = I_h1 * np.ones(N_iter)         # stronger driving current to E1s
I_h2 = 180                          # driving current to non-autaptic neuron
I2 = I_h2 * np.ones(N_iter)         # weaker current to E2

I_inj = np.array([I1, I2])
I_excTA = TimedArray(values = I_inj.T, dt = dt*ms, name = 'I_excTA')

# convert to brian2 with the standard autapse model
E, aut_E = batch_E.meetBrian(stimulus_name = I_excTA, synapse_eq = syn_eq)

''' - - - define the inhibitory neuron - - - '''
param_I = [I_neuron]
x_start = np.array([[-60, 0, 0]])
t_start = np.array([0.])

# create batch 
batch_I = batchAQUA(param_I)
batch_I.Initialise(x_start, t_start)

# input current will be just subthreshold
threshold, _ = batch_I.get_threshold(idx = 0)
I_inh = np.array([threshold*np.ones(N_iter)])
I_inhTA = TimedArray(values = I_inh.T, dt = dt*ms, name = 'I_inhTA')

# create brian objects, no effective autapse here.
I, aut_I = batch_I.meetBrian(stimulus_name = I_inhTA, synapse_eq = syn_eq)


''' - - - create the synapses - - - '''
# create the synapses between E1 and E2...
# I_syn represents the synapse strength
#syn_eq = """
#dI_syn/dt = -(I_syn/t_syn)/ms : 1 (clock-driven)
#t_syn : 1
#w_syn : 1
#g_total_syn = I_syn : 1 
#"""
model_exc = '''w_exc : 1'''
model_inh = '''w_inh : 1'''
syn_on_pre_exc = '''Syn_exc += w_exc'''     # excitatory presynaptic neuron
syn_on_pre_inh = '''Syn_inh += w_inh'''     # inhibitory presynaptic neuron

""" - - STDP - - """
w_max = 150 # maximum allowed current through an inhibitory synapse
taupre = taupost = 20 * ms
dApre = 20      # the maximum change in the weight in one step
dApost = -dApre * (taupre/taupost) * 1.05


model_stdp = '''
w_inh : 1
w_exc : 1
dApre/dt = -Apre / taupre : 1 (event-driven)
dApost/dt = -Apost / taupost : 1 (event-driven)
'''
# if pre- is inhibitory
on_pre_stdp_inh = '''
Syn_inh += w_inh
Apre += dApre
w_inh = clip(w_inh + Apost, -w_max, 0)
''' 
# if pre- is excitatory
on_pre_stdp_exc = '''
Syn_exc += w_exc
Apre += dApre
w_exc = clip(w_exc + Apost, 0, w_max)
''' 
on_post_stdp = '''
Apost += dApost
w_inh = clip(w_inh + Apre, 0, w_max)
w_exc = clip(w_exc + Apre, 0, w_max)
'''

'''exc. synapses'''
# fully connect excitatory neurons (ignoring autapses)
syn_E = Synapses(E, E, 
                model = model_exc,
                on_pre = syn_on_pre_exc,
                method = 'rk2')
syn_E.connect(condition = 'i != j')     # fully connected minus autapses

## Set exc. synapse variables here...
E.Syn_exc = 0            # pA
E.t_exc = 5              # ms
syn_E.w_exc[0, 1] = 60    # pA, weight from E1 -> E2
syn_E.w_exc[1, 0] = 60    # pA, weight from E2 -> E1

'''inh. to exc. synapses'''
syn_IE = Synapses(I, E,
                model = model_stdp,
                on_pre = on_pre_stdp_inh,
                on_post = on_post_stdp,
                method = 'rk2')
syn_IE.connect()         # fully connected to the 2 exc. neurons

## set inh. synapse variables for post-synaptic population
E.Syn_inh = 0
E.t_inh = 5                # ms
syn_IE.w_inh[0, 0] = -50   # pA, weight from I -> E1
syn_IE.w_inh[0, 1] = -50   # pA, weight from I -> E2
# alt., syn_I.w_syn[0, :] = 50, as they are equal

'''exc. to inh. synapses'''
syn_EI = Synapses(E, I,
                model = model_stdp,
                on_pre = on_pre_stdp_exc,
                on_post = on_post_stdp,
                method = 'rk2')
syn_EI.connect()         # both excitatory neurons connect to I

## set inh. synapse variables for post-syn population
I.Syn_exc = 0.            # initially no current through the synapse
I.t_exc = 5               # ms
I.t_inh = 5
syn_EI.w_exc[0, 0] = 50   # pA, weight from I -> E1
syn_EI.w_exc[1, 0] = 50   # pA, weight from I -> E2
# alt., syn_I.w_syn[0, :] = 50, as they are equal

''' - - simulation - - '''
# set simulation parameters
defaultclock.dt = dt*ms
# Monitors
M_v_E_aut = StateMonitor(E, ['v', 'Syn_exc', 'Syn_inh', 'w'], record = True)
M_v_I_aut = StateMonitor(I, ['v', 'Syn_exc', 'Syn_inh'], record = True)
spikemon_E_aut = SpikeMonitor(E, record = True)
spikemon_I_aut = SpikeMonitor(I, record = True)
# M_syn_E = StateMonitor(syn_E, 'Syn_exc', record = True)   # record the current through the synapse
# M_syn_IE = StateMonitor(syn_IE, 'Syn_inh', record = True)   
# M_syn_EI = StateMonitor(syn_EI, 'Syn_exc', record = True)   

# create networks
net = Network(E, I, aut_E, aut_I, syn_E, syn_IE, syn_EI, M_v_E, M_v_I, spikemon_E, spikemon_I) #, M_syn_E, M_syn_IE, M_syn_EI)

net.run(T*ms)

spike_trains_E = spikemon_E_aut.spike_trains()
spike_trains_I = spikemon_I_aut.spike_trains()